# Train & Export Stockfish-Distilled 3-Layer Quantized Value CNN (`nojs/chess`)

This notebook builds a diverse chess position dataset labeled by **Stockfish**, trains a lightweight **BatchNorm-free 3-Layer Quantized Convolutional Value Network (`SVGChessValueCNN`, `1,366` parameters)** with **8-bit Straight-Through Quantization-Aware Training (`ste_q8`)**, verifies bit-exact parity against a pure-NumPy HTML5 SVG `<filter>` simulator + 2-ply Minimax tactical benchmark, and exports `weights.json` for `chess/`.

### Architecture & SVG Hardware Co-Design
1. **Orientation-Normalized Signed 6-Channel Board Input (`2` RGB Tiles around `Z0 = 128/255`)**:
   - Ranks are normalized to the **AI's perspective** (`ar = r` when AI is White, `7 - r` when AI is Black; at every depth-2 leaf $s_2$ after `AI move -> Opponent reply`, it is the AI's turn to move).
   - `RGB_0 = (Pawn, Knight, Bishop)` and `RGB_1 = (Rook, Queen, King)` with **AI piece = `255/255`**, **Empty / 1px Moat = `128/255` (`Z0`)**, and **Opponent piece = `1/255`**.
2. **3 Convolutional Stages + Lossless Linear Material/Phase Skip (`1,366` Params, `0` BatchNorm)**:
   - **Layer 1 (`8×8 → 4×4`)**: `SVGPointwise1x1(6 → 15)` (`<feColorMatrix>`) + `3×3` depthwise `<feConvolveMatrix>` (`padding=1`) + `2×2` AvgPool.
   - **Layer 2 (`4×4 → 2×2`)**: `SVGPointwise1x1(15 → 18)` + `3×3` depthwise `<feConvolveMatrix>` (`padding=1`) + `2×2` AvgPool.
   - **Layer 3 (`2×2 → 1×1`)**: `SVGPointwise1x1(18 → 24)` + `2×2` global depthwise `<feConvolveMatrix>`.
   - **Lossless Linear Skip (`8×8 → 1×1`, `3` ch)**: `6 → 3` `<feColorMatrix>` around `Z0` aggregated across `8×8 → 4×4 → 2×2 → 1×1` via centered `2×2` sums (`divisor = 1.25, 1.6, 1.0`), preserving `~2.8 LSB` per single pawn without ReLU clipping.
   - **Calibrated Value Head (`27 → 1`)**: `9` RGB `<feColorMatrix>` triplets summed around `Z0 = 128/255` + 256-entry `<feComponentTransfer type="table">` Sigmoid LUT mapping logits into `[3/255 .. 253/255]` (`0.50` WDL = exact `128/255 = Z0` draw; `0` = invalid move, `254` = Mate-in-1, `255` = empty reply mask for `<feMorphology operator="erode">`).

**Usage in Google Colab**: Select a **T4 GPU** runtime and click **Runtime $\rightarrow$ Run all** (~3–5 minutes end-to-end) to install Stockfish, generate & label positions, train with 8-bit QAT, verify SVG 2-ply Minimax tactics, and automatically download `weights.json`.


In [ ]:
# ---------------------------------------------------------------------------
# 0. Install Stockfish Binary & python-chess on Google Colab
# ---------------------------------------------------------------------------
import os
import platform
import shutil
import stat
import subprocess
import sys
import tarfile
import urllib.request

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "chess"])

def ensure_stockfish() -> str:
    """Locate or install a fast Stockfish UCI binary on Colab / Linux / macOS."""
    for candidate in (shutil.which("stockfish"), "/usr/games/stockfish", "/usr/local/bin/stockfish", "./stockfish_bin"):
        if candidate and os.path.isfile(candidate) and os.access(candidate, os.X_OK):
            return candidate

    # 1. Fast Ubuntu/Debian apt install on Google Colab (~4s)
    if shutil.which("apt-get"):
        print("Installing Stockfish via apt-get...")
        res = subprocess.run(
            ["apt-get", "update", "-qq"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        if res.returncode == 0:
            subprocess.run(
                ["apt-get", "install", "-y", "-qq", "stockfish"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
        for candidate in (shutil.which("stockfish"), "/usr/games/stockfish"):
            if candidate and os.path.isfile(candidate):
                return candidate

    # 2. Direct official GitHub release binary fallback for Linux x86_64
    if platform.system() == "Linux" and platform.machine() in ("x86_64", "AMD64"):
        print("Downloading official Stockfish Linux x86-64 AVX2 release...")
        url = "https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-ubuntu-x86-64-avx2.tar"
        tar_path = "stockfish_avx2.tar"
        urllib.request.urlretrieve(url, tar_path)
        with tarfile.open(tar_path, "r") as tf:
            for member in tf.getmembers():
                if member.isfile() and "stockfish" in os.path.basename(member.name) and not member.name.endswith(".tar"):
                    extracted = tf.extractfile(member)
                    if extracted is not None:
                        with open("./stockfish_bin", "wb") as out_f:
                            out_f.write(extracted.read())
                        os.chmod("./stockfish_bin", os.stat("./stockfish_bin").st_mode | stat.S_IEXEC)
                        return "./stockfish_bin"

    raise RuntimeError("Could not find or install Stockfish binary!")

STOCKFISH_PATH = ensure_stockfish()
print(f"Stockfish ready at: {STOCKFISH_PATH}")


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
import json
import math
import random
import time
from typing import Dict, List, Optional, Sequence, Tuple

import chess
import chess.engine
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Exact 8-bit integer zero-point: 128 / 255 avoids the 127.5 half-integer rounding drift!
ZERO_POINT_U8 = 128
ZERO_POINT = round(ZERO_POINT_U8 / 255.0, 6)  # 0.501961
POS_VAL_U8 = 255                              # +127/255 above Z0 (AI piece)
NEG_VAL_U8 = 1                                # -127/255 below Z0 (Opponent piece)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | Device: {device} | CPU count: {os.cpu_count()} | Z0 = {ZERO_POINT} ({ZERO_POINT_U8}/255)")


In [ ]:
# ---------------------------------------------------------------------------
# 1. Orientation-Normalized 6-Channel Board Encoding & Stratified Position Sampler
# ---------------------------------------------------------------------------

PIECE_TYPES = (chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING)
PIECE_VAL_QUICK = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}

def encode_board_uint8(board: chess.Board, ai_color: Optional[chess.Color] = None) -> np.ndarray:
    """Encode `board` into uint8 `(6, 8, 8)` from `ai_color`'s perspective (defaults to `board.turn`).
    - Axis 0 (channel 0..5): Pawn, Knight, Bishop, Rook, Queen, King
    - Axis 1 (ar 0..7): AI back rank (0) -> Opponent back rank (7)
    - Axis 2 (af 0..7): File a (0) -> File h (7)
    Values are exact bytes `{1, 128, 255}` (`NEG_VAL_U8`, `ZERO_POINT_U8`, `POS_VAL_U8`).
    """
    persp = board.turn if ai_color is None else ai_color
    arr = np.full((6, 8, 8), ZERO_POINT_U8, dtype=np.uint8)
    for sq, pc in board.piece_map().items():
        r, f = divmod(sq, 8)
        ar = r if persp == chess.WHITE else (7 - r)
        ch = pc.piece_type - 1
        arr[ch, ar, f] = POS_VAL_U8 if pc.color == persp else NEG_VAL_U8
    return arr


def cp_to_wdl(cp: float, scale: float = 380.0) -> float:
    """Map Stockfish centipawns (from side-to-move's perspective) to smooth WDL win probability in [0.005, 0.995]."""
    x = max(-18.0, min(18.0, cp / scale))
    w = 1.0 / (1.0 + math.exp(-x))
    return float(min(0.995, max(0.005, w)))


def _pick_semi_smart_move(board: chess.Board, rng: random.Random, tactical_bias: float = 0.55) -> chess.Move:
    """Fast exploratory policy mixing captures/checks/promotions with quiet development moves."""
    moves = list(board.legal_moves)
    if rng.random() < tactical_bias:
        tactical = [m for m in moves if board.is_capture(m) or m.promotion is not None or board.gives_check(m)]
        if tactical:
            # Prefer capturing higher-value targets (MVV-LVA flavor) with randomness
            tactical.sort(
                key=lambda m: PIECE_VAL_QUICK.get(board.piece_type_at(m.to_square) or chess.PAWN, 1) + rng.random() * 2.5,
                reverse=True,
            )
            return tactical[0]
    return rng.choice(moves)


def _synthesize_endgame_board(rng: random.Random) -> Optional[chess.Board]:
    """Construct valid random endgame boards (K+P, K+R, K+Q, minor-piece & rook endgames)."""
    templates = [
        ([chess.QUEEN], []),
        ([chess.ROOK], []),
        ([chess.PAWN, chess.PAWN], [chess.PAWN]),
        ([chess.ROOK, chess.PAWN], [chess.BISHOP, chess.PAWN]),
        ([chess.BISHOP, chess.KNIGHT, chess.PAWN], [chess.ROOK]),
        ([chess.QUEEN, chess.PAWN], [chess.ROOK, chess.KNIGHT]),
        ([chess.KNIGHT, chess.PAWN, chess.PAWN], [chess.BISHOP, chess.PAWN]),
        ([], []),
    ]
    strong_pcs, weak_pcs = rng.choice(templates)
    if rng.random() < 0.5:
        w_pcs, b_pcs = strong_pcs, weak_pcs
    else:
        w_pcs, b_pcs = weak_pcs, strong_pcs

    for _ in range(30):
        b = chess.Board.empty()
        sqs = rng.sample(range(64), 2 + len(w_pcs) + len(b_pcs))
        wk, bk = sqs[0], sqs[1]
        if chess.square_distance(wk, bk) <= 1:
            continue
        b.set_piece_at(wk, chess.Piece(chess.KING, chess.WHITE))
        b.set_piece_at(bk, chess.Piece(chess.KING, chess.BLACK))
        ok = True
        idx = 2
        for pt in w_pcs:
            sq = sqs[idx]; idx += 1
            if pt == chess.PAWN and chess.square_rank(sq) in (0, 7):
                ok = False; break
            b.set_piece_at(sq, chess.Piece(pt, chess.WHITE))
        if not ok:
            continue
        for pt in b_pcs:
            sq = sqs[idx]; idx += 1
            if pt == chess.PAWN and chess.square_rank(sq) in (0, 7):
                ok = False; break
            b.set_piece_at(sq, chess.Piece(pt, chess.BLACK))
        if not ok:
            continue
        b.turn = chess.WHITE if rng.random() < 0.5 else chess.BLACK
        if b.is_valid():
            return b
    return None


def generate_diverse_position_pairs(
    target_singletons: int = 64000,
    target_sibling_pairs: int = 16000,
    seed: int = SEED,
) -> Tuple[List[str], List[Tuple[str, str]], List[str]]:
    """Build a deduplicated corpus of:
      - `single_fens`: diverse openings, tactical middlegames, imbalances, & endgames
      - `sibling_pairs`: `(fen_a, fen_b)` leaf pairs arising from 2 distinct moves on the same parent board
        (trains pairwise ranking margin so 2-ply Minimax orders sibling moves like Stockfish!)
      - `regimes`: category tag per `single_fen` for stratified validation/test diagnostics
    """
    rng = random.Random(seed)
    seen_keys = set()
    single_fens: List[str] = []
    regimes: List[str] = []
    sibling_pairs: List[Tuple[str, str]] = []

    def _add_fen(bd: chess.Board, reg: str) -> bool:
        key = bd.board_fen() + (" w" if bd.turn == chess.WHITE else " b")
        if key in seen_keys:
            return False
        seen_keys.add(key)
        single_fens.append(bd.fen())
        regimes.append(reg)
        return True

    # 1. Endgames (~18% of singletons)
    n_end_target = int(0.18 * target_singletons)
    while len(single_fens) < n_end_target:
        eb = _synthesize_endgame_board(rng)
        if eb is not None:
            _add_fen(eb, "endgame")

    # 2. Game rollouts covering openings (plies 2..14), tactical captures/checks, & middlegames/endgames
    while len(single_fens) < target_singletons or len(sibling_pairs) < target_sibling_pairs:
        b = chess.Board()
        max_plies = rng.randint(18, 95)
        tac_bias = rng.choice([0.20, 0.50, 0.75])
        for ply in range(max_plies):
            if b.is_game_over():
                if len(single_fens) < target_singletons:
                    _add_fen(b, "terminal")
                break

            leg = list(b.legal_moves)
            # Sample sibling leaf pairs `(s -> m_a, s -> m_b)` (both evaluated from the SAME turn!)  
            if len(leg) >= 3 and len(sibling_pairs) < target_sibling_pairs and rng.random() < 0.16:
                caps = [m for m in leg if b.is_capture(m) or b.gives_check(m) or m.promotion is not None]
                quiets = [m for m in leg if m not in caps]
                if caps and quiets:
                    ma, mb = rng.choice(caps), rng.choice(quiets)
                else:
                    ma, mb = rng.sample(leg, 2)
                b.push(ma)
                fa = b.fen()
                b.pop()
                b.push(mb)
                fb = b.fen()
                b.pop()
                sibling_pairs.append((fa, fb))

            mv = _pick_semi_smart_move(b, rng, tactical_bias=tac_bias)
            was_cap_or_chk = b.is_capture(mv) or b.gives_check(mv) or (mv.promotion is not None)
            b.push(mv)

            if len(single_fens) < target_singletons:
                if ply < 14 and rng.random() < 0.32:
                    _add_fen(b, "opening")
                elif was_cap_or_chk and rng.random() < 0.45:
                    _add_fen(b, "tactical")
                elif ply >= 50 and len(b.piece_map()) <= 10 and rng.random() < 0.40:
                    _add_fen(b, "endgame")
                elif rng.random() < 0.22:
                    _add_fen(b, "middlegame")

    return single_fens, sibling_pairs, regimes


t0 = time.time()
SINGLE_FENS, SIBLING_PAIRS, REGIMES = generate_diverse_position_pairs(
    target_singletons=64000,
    target_sibling_pairs=16000,
)
print(
    f"Generated {len(SINGLE_FENS):,} unique positions + {len(SIBLING_PAIRS):,} sibling move pairs "
    f"({len(SINGLE_FENS) + 2 * len(SIBLING_PAIRS):,} total boards) in {time.time() - t0:.1f}s"
)
for reg in sorted(set(REGIMES)):
    print(f"  Regime {reg:11s}: {REGIMES.count(reg):6,d} ({100.0 * REGIMES.count(reg) / len(REGIMES):4.1f}%)")


In [ ]:
# ---------------------------------------------------------------------------
# 2. Parallel Persistent-UCI Stockfish Distillation Labeling
# ---------------------------------------------------------------------------

SF_NODES_LIMIT = 1500  # ~1,500 NNUE nodes/pos (~0.5ms/pos on Stockfish, sharp tactical & positional eval)

def label_fens_with_stockfish(fens: Sequence[str], sf_path: str, nodes_limit: int = SF_NODES_LIMIT) -> Tuple[np.ndarray, np.ndarray]:
    """Evaluate `fens` using persistent Stockfish UCI workers (releases GIL over pipes!).
    Returns `(encoded_boards_u8 (N, 6, 8, 8), target_wdl_f32 (N,))` from `board.turn`'s perspective.
    """
    n = len(fens)
    x_u8 = np.empty((n, 6, 8, 8), dtype=np.uint8)
    y_wdl = np.empty(n, dtype=np.float32)
    num_workers = max(2, min(4, (os.cpu_count() or 2)))
    chunk_size = math.ceil(n / num_workers)

    def _worker(w_id: int, start: int, end: int) -> None:
        eng = chess.engine.SimpleEngine.popen_uci(sf_path)
        try:
            eng.configure({"Threads": 1, "Hash": 16})
            lim = chess.engine.Limit(nodes=nodes_limit)
            for idx in range(start, end):
                bd = chess.Board(fens[idx])
                x_u8[idx] = encode_board_uint8(bd, ai_color=bd.turn)
                if bd.is_checkmate():
                    y_wdl[idx] = 0.005  # Side to move is checkmated!
                elif bd.is_stalemate() or bd.is_insufficient_material():
                    y_wdl[idx] = 0.500
                else:
                    info = eng.analyse(bd, lim)
                    pov_score = info["score"].pov(bd.turn)
                    mate_n = pov_score.mate()
                    if mate_n is not None:
                        y_wdl[idx] = 0.995 if mate_n > 0 else 0.005
                    else:
                        cp = float(pov_score.score() or 0.0)
                        y_wdl[idx] = cp_to_wdl(cp)
                if w_id == 0 and ((idx - start + 1) % 5000 == 0):
                    pct = 100.0 * (idx - start + 1) / max(1, end - start)
                    print(f"  [Worker 0] Labeled {idx - start + 1:,}/{end - start:,} ({pct:.0f}%)...")
        finally:
            eng.quit()

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futs = []
        for w_id in range(num_workers):
            s_i = w_id * chunk_size
            e_i = min(n, (w_id + 1) * chunk_size)
            if s_i < e_i:
                futs.append(pool.submit(_worker, w_id, s_i, e_i))
        for fut in futs:
            fut.result()

    elapsed = time.time() - t0
    print(f"Stockfish labeled {n:,} positions in {elapsed:.1f}s ({n / max(elapsed, 1e-3):,.0f} pos/sec across {num_workers} UCI workers)")
    return x_u8, y_wdl


all_pair_fens = [f for pair in SIBLING_PAIRS for f in pair]
combined_fens = SINGLE_FENS + all_pair_fens
x_all_u8, y_all_wdl = label_fens_with_stockfish(combined_fens, STOCKFISH_PATH, nodes_limit=SF_NODES_LIMIT)

n_sing = len(SINGLE_FENS)
x_sing_u8, y_sing_wdl = x_all_u8[:n_sing], y_all_wdl[:n_sing]
x_pairs_u8 = x_all_u8[n_sing:].reshape(len(SIBLING_PAIRS), 2, 6, 8, 8)
# Note: sibling boards have the OPPONENT to move (`not parent.turn`), so from the parent mover's
# perspective, parent win-prob is `1.0 - y_opponent_wdl`, and `y_sing_wdl` directly trains `board.turn` WDL!
y_pairs_wdl = y_all_wdl[n_sing:].reshape(len(SIBLING_PAIRS), 2)

# Stratified 80% Train / 10% Val / 10% Test split on singletons + 90/10 split on sibling pairs
rng_np = np.random.default_rng(SEED)
perm = rng_np.permutation(n_sing)
n_train = int(0.80 * n_sing)
n_val = int(0.10 * n_sing)
tr_idx, va_idx, te_idx = perm[:n_train], perm[n_train : n_train + n_val], perm[n_train + n_val :]

x_train_f = x_sing_u8[tr_idx].astype(np.float32) / 255.0
y_train_f = y_sing_wdl[tr_idx]
x_val_f = x_sing_u8[va_idx].astype(np.float32) / 255.0
y_val_f = y_sing_wdl[va_idx]
x_test_f = x_sing_u8[te_idx].astype(np.float32) / 255.0
y_test_f = y_sing_wdl[te_idx]
regimes_test = [REGIMES[i] for i in te_idx]

n_pair_tr = int(0.90 * len(SIBLING_PAIRS))
xp_train_f = x_pairs_u8[:n_pair_tr].astype(np.float32) / 255.0
yp_train_f = y_pairs_wdl[:n_pair_tr]
xp_val_f = x_pairs_u8[n_pair_tr:].astype(np.float32) / 255.0
yp_val_f = y_pairs_wdl[n_pair_tr:]

BATCH_SIZE = 256
train_loader = DataLoader(TensorDataset(torch.from_numpy(x_train_f), torch.from_numpy(y_train_f)), batch_size=BATCH_SIZE, shuffle=True)
pair_loader = DataLoader(TensorDataset(torch.from_numpy(xp_train_f), torch.from_numpy(yp_train_f)), batch_size=128, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(x_val_f), torch.from_numpy(y_val_f)), batch_size=512, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.from_numpy(x_test_f), torch.from_numpy(y_test_f)), batch_size=512, shuffle=False)

plt.figure(figsize=(7.5, 2.8))
plt.hist(y_train_f, bins=50, color="#38bdf8", edgecolor="#0f172a", alpha=0.85)
plt.axvline(0.5, color="#fbbf24", linestyle="--", label="Equal (0.0 cp)")
plt.title("Stockfish Distilled Target WDL Distribution (Train Split)")
plt.xlabel("Target Win Probability (AI Perspective)")
plt.ylabel("Positions")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 3. BatchNorm-Free 3-Layer Quantized Value CNN (`SVGChessValueCNN`)
#    Co-Designed for Bit-Exact HTML5 SVG `<feConvolveMatrix>` & `<feColorMatrix>`
# ---------------------------------------------------------------------------

def ste_q8(x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
    """Exact SVG intermediate RGBA8888 buffer physics: clamp to [0, 1] + 8-bit uint8 STE rounding."""
    xc = torch.clamp(x, 0.0, 1.0)
    if not quantize:
        return xc
    xq = torch.round(xc * 255.0) / 255.0
    return xc + (xq - xc).detach()


class SVGPointwise1x1(nn.Module):
    """Pointwise 1x1 cross-channel mixing matching SVG `<feColorMatrix>` RGB triplets
    summed around exact integer zero-point `Z0 = 128/255` via `<feComposite k2=1 k3=1 k4=-Z0>`,
    followed by per-channel `<feComponentTransfer type='linear'>` rescaling into `[0, 1]`.
    """

    def __init__(self, in_channels: int, out_channels: int, init_gain: float = 1.0, center_input: bool = False):
        super().__init__()
        assert in_channels % 3 == 0 and out_channels % 3 == 0
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels))
        self.bias = nn.Parameter(torch.zeros(out_channels))
        nn.init.kaiming_normal_(self.weight, mode="fan_in", nonlinearity="relu")
        with torch.no_grad():
            self.weight.mul_(init_gain)
            if center_input:
                # Center bias around Z0 so empty squares (x = Z0) start active in linear regime (+0.06)
                self.bias.copy_(0.06 - self.weight.sum(dim=1) * ZERO_POINT)
            else:
                self.bias.fill_(0.03)

    def compute_channel_scales(self) -> torch.Tensor:
        with torch.no_grad():
            w_pos = torch.clamp(self.weight, min=0.0).sum(dim=1)
            w_neg = torch.clamp(-self.weight, min=0.0).sum(dim=1)
            return torch.clamp(torch.maximum(w_pos / 0.44, w_neg / 0.45), min=1.0)

    def forward(self, x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
        g_in = self.in_channels // 3
        z0 = ZERO_POINT
        s_ch = self.compute_channel_scales()
        s_col = s_ch.unsqueeze(1)
        s_view = s_ch.view(1, self.out_channels, 1, 1)

        w0 = (self.weight[:, 0:3] / s_col).view(self.out_channels, 3, 1, 1)
        acc = ste_q8(z0 + F.conv2d(x[:, 0:3], w0), quantize=quantize)
        for g in range(1, g_in):
            wg = (self.weight[:, 3 * g : 3 * (g + 1)] / s_col).view(self.out_channels, 3, 1, 1)
            part = ste_q8(z0 + F.conv2d(x[:, 3 * g : 3 * (g + 1)], wg), quantize=quantize)
            acc = ste_q8(acc + part - z0, quantize=quantize)

        b_view = self.bias.view(1, self.out_channels, 1, 1)
        return ste_q8(s_view * acc + (b_view - z0 * s_view), quantize=quantize)


class SVGLinearSkip6to3(nn.Module):
    """Lossless centered 6 -> 3 piece/material projection around Z0 = 128/255 aggregated
    across 8x8 -> 4x4 -> 2x2 -> 1x1 via centered 2x2 `<feConvolveMatrix>` sums (divisor = 1.25, 1.6, 1.0),
    preserving ~2.8 LSB per single pawn globally without ReLU clipping!
    """

    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(3, 6))
        with torch.no_grad():
            # Channel 0: Classical piece values (P=1, N=3.05, B=3.3, R=5, Q=9, K=0) -> ~2.8 LSB/pawn at 1x1
            self.weight[0] = torch.tensor([0.044, 0.135, 0.145, 0.220, 0.396, 0.0])
            # Channel 1: Minor/Major piece balance vs Pawns
            self.weight[1] = torch.tensor([-0.055, 0.120, 0.135, 0.075, 0.180, 0.0])
            # Channel 2: Heavy-piece (R/Q) presence / tactical danger
            self.weight[2] = torch.tensor([0.020, 0.045, 0.045, 0.195, 0.340, 0.0])

    def effective_weight(self) -> torch.Tensor:
        # Bound per-row L1 norm <= 0.95 and per-piece weight <= 0.44 so 2x2 sums around Z0 never clip [0, 1]
        w = torch.clamp(self.weight, -0.44, 0.44)
        l1 = w.abs().sum(dim=1, keepdim=True).clamp(min=0.95) / 0.95
        return w / l1

    def forward(self, x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
        z0 = ZERO_POINT
        w = self.effective_weight()  # (3, 6) applied to (x - z0)
        # Group 0 (P, N, B): z0 + w[:, 0:3] @ (x[:, 0:3] - z0)
        w0 = w[:, 0:3].view(3, 3, 1, 1)
        off0 = (z0 - w[:, 0:3].sum(dim=1) * z0).view(1, 3, 1, 1)
        p0 = ste_q8(F.conv2d(x[:, 0:3], w0) + off0, quantize=quantize)

        # Group 1 (R, Q, K): z0 + w[:, 3:6] @ (x[:, 3:6] - z0)
        w1 = w[:, 3:6].view(3, 3, 1, 1)
        off1 = (z0 - w[:, 3:6].sum(dim=1) * z0).view(1, 3, 1, 1)
        p1 = ste_q8(F.conv2d(x[:, 3:6], w1) + off1, quantize=quantize)

        s8 = ste_q8(p0 + p1 - z0, quantize=quantize)                                      # (B, 3, 8, 8)
        # Step 1 (8x8 -> 4x4): sum_2x2 / 1.25 - 2.2*z0  (4/1.25 - 2.2 == 1.0 exact)
        s4 = ste_q8((4.0 / 1.25) * F.avg_pool2d(s8, 2, 2) - round(2.2 * z0, 6), quantize=quantize)
        # Step 2 (4x4 -> 2x2): sum_2x2 / 1.60 - 1.5*z0  (4/1.60 - 1.5 == 1.0 exact)
        s2 = ste_q8((4.0 / 1.60) * F.avg_pool2d(s4, 2, 2) - round(1.5 * z0, 6), quantize=quantize)
        # Step 3 (2x2 -> 1x1): sum_2x2 / 1.00 - 3.0*z0  (4/1.00 - 3.0 == 1.0 exact)
        s1 = ste_q8(4.0 * F.avg_pool2d(s2, 2, 2) - round(3.0 * z0, 6), quantize=quantize)
        return s1


class SVGChessValueCNN(nn.Module):
    """3-Layer Quantized Chess Value Network (`1,366` params, `0` BatchNorm) for SVG 2-Ply Minimax."""

    def __init__(self):
        super().__init__()
        # Layer 1: 6 -> 15 (5 RGB tiles) pointwise + 3x3 depthwise (padding=1) + 2x2 avg pool (8x8 -> 4x4)
        self.mix1 = SVGPointwise1x1(6, 15, init_gain=1.1, center_input=True)
        self.dwconv1 = nn.Conv2d(15, 15, kernel_size=3, stride=1, padding=1, groups=15, bias=True)

        # Layer 2: 15 -> 18 (6 RGB tiles) pointwise + 3x3 depthwise (padding=1) + 2x2 avg pool (4x4 -> 2x2)
        self.mix2 = SVGPointwise1x1(15, 18, init_gain=1.15)
        self.dwconv2 = nn.Conv2d(18, 18, kernel_size=3, stride=1, padding=1, groups=18, bias=True)

        # Layer 3: 18 -> 24 (8 RGB tiles) pointwise + 2x2 global depthwise aggregator (2x2 -> 1x1)
        self.mix3 = SVGPointwise1x1(18, 24, init_gain=1.15)
        self.dwconv3 = nn.Conv2d(24, 24, kernel_size=2, stride=1, padding=0, groups=24, bias=True)

        # Lossless linear global material skip (3 channels = 1 RGB tile)
        self.skip_mat = SVGLinearSkip6to3()

        # Scalar Value Head: 24 + 3 = 27 features (9 RGB triplets) -> 1 logit
        self.fc = nn.Linear(27, 1, bias=True)

        for m in (self.dwconv1, self.dwconv2, self.dwconv3):
            nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
            with torch.no_grad():
                m.bias.fill_(0.02)
        with torch.no_grad():
            self.fc.weight.zero_()
            self.fc.weight[0, 24] = 22.0  # Warm-start centered on linear classical material channel (~0.24 logits/pawn)
            self.fc.bias.zero_()

    def compute_logit_half_range(self) -> torch.Tensor:
        with torch.no_grad():
            w_conv = self.fc.weight[:, :24]
            w_skip_l1 = self.fc.weight[:, 24:].abs().sum(dim=1) * 0.502
            w_pos = torch.clamp(w_conv, min=0.0).sum(dim=1) + w_skip_l1 + torch.clamp(self.fc.bias, min=0.0)
            w_neg = torch.clamp(-w_conv, min=0.0).sum(dim=1) + w_skip_l1 + torch.clamp(-self.fc.bias, min=0.0)
            m_req = torch.maximum(w_pos / (2.0 * 0.44), w_neg / (2.0 * 0.45)).amax()
            return torch.clamp(m_req, min=4.5)

    def forward(self, x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
        skip = self.skip_mat(x, quantize=quantize)                          # (B, 3, 1, 1) centered at Z0

        h = self.mix1(x, quantize=quantize)                                 # (B, 15, 8, 8)
        h = ste_q8(self.dwconv1(h), quantize=quantize)                      # (B, 15, 8, 8)
        h = ste_q8(F.avg_pool2d(h, 2, 2), quantize=quantize)                # (B, 15, 4, 4)

        h = self.mix2(h, quantize=quantize)                                 # (B, 18, 4, 4)
        h = ste_q8(self.dwconv2(h), quantize=quantize)                      # (B, 18, 4, 4)
        h = ste_q8(F.avg_pool2d(h, 2, 2), quantize=quantize)                # (B, 18, 2, 2)

        h = self.mix3(h, quantize=quantize)                                 # (B, 24, 2, 2)
        h = ste_q8(self.dwconv3(h), quantize=quantize)                      # (B, 24, 1, 1)

        z0 = ZERO_POINT
        if not quantize:
            feat_c = torch.cat([h.flatten(1), skip.flatten(1) - z0], dim=1) # (B, 27)
            return self.fc(feat_c).squeeze(-1)

        feat = torch.cat([h, skip], dim=1).flatten(1)                       # (B, 27)
        m_half = self.compute_logit_half_range()
        two_m = 2.0 * m_half
        n_groups = feat.shape[1] // 3
        acc = ste_q8(
            z0 + (self.fc.bias.unsqueeze(0) + feat[:, 0:3] @ self.fc.weight[:, 0:3].t()) / two_m,
            quantize=True,
        )
        for g in range(1, n_groups):
            wg = self.fc.weight[:, 3 * g : 3 * (g + 1)]
            if g == n_groups - 1:
                # Skip triplet (g == 8) is centered around Z0!
                part = ste_q8(z0 + ((feat[:, 3 * g : 3 * (g + 1)] - z0) @ wg.t()) / two_m, quantize=True)
            else:
                part = ste_q8(z0 + (feat[:, 3 * g : 3 * (g + 1)] @ wg.t()) / two_m, quantize=True)
            acc = ste_q8(acc + part - z0, quantize=True)
        return ((acc - z0) * two_m).squeeze(-1)


model = SVGChessValueCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"SVGChessValueCNN initialized | Total trainable parameters: {total_params:,} (0 BatchNorm layers)")


In [ ]:
# ---------------------------------------------------------------------------
# 4. Train with 8-Bit STE QAT (WDL Calibration + Sibling Move-Ranking Loss)
# ---------------------------------------------------------------------------

EPOCHS = 26
optimizer = torch.optim.AdamW(model.parameters(), lr=3.5e-3, weight_decay=1.5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=6e-5)

@torch.no_grad()
def evaluate_value_net(loader: DataLoader, pairs_x: np.ndarray, pairs_y: np.ndarray, quantize: bool = True):
    model.eval()
    preds_list, tgts_list = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        probs = torch.sigmoid(model(xb, quantize=quantize)).cpu().numpy()
        preds_list.append(probs)
        tgts_list.append(yb.numpy())
    preds = np.concatenate(preds_list)
    tgts = np.concatenate(tgts_list)
    mae = float(np.mean(np.abs(preds - tgts)))
    corr = float(np.corrcoef(preds, tgts)[0, 1])

    # Pairwise move-ordering accuracy on sibling positions where Stockfish differs by >= 0.08 WDL (~30 cp)
    px = torch.from_numpy(pairs_x.reshape(-1, 6, 8, 8)).to(device)
    p_out = torch.sigmoid(model(px, quantize=quantize)).cpu().numpy().reshape(-1, 2)
    diff_true = pairs_y[:, 0] - pairs_y[:, 1]
    mask_sig = np.abs(diff_true) >= 0.08
    pair_acc = float(np.mean(np.sign(p_out[mask_sig, 0] - p_out[mask_sig, 1]) == np.sign(diff_true[mask_sig]))) if mask_sig.any() else 0.0
    return mae, corr, pair_acc


history = {"train_loss": [], "val_mae": [], "val_corr": [], "val_pair_acc": []}
best_score = -1e9
best_state = None
best_metrics = (1.0, 0.0, 0.0)

for epoch in range(1, EPOCHS + 1):
    model.train()
    use_qat = (epoch >= 3)
    run_loss, steps = 0.0, 0
    pair_iter = iter(pair_loader)

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()

        logits = model(xb, quantize=use_qat)
        probs = torch.sigmoid(logits)
        # Smooth BCE + Huber WDL calibration
        loss_wdl = F.binary_cross_entropy_with_logits(logits, yb) + 0.5 * F.smooth_l1_loss(probs, yb, beta=0.05)

        # Sibling move pairwise ranking loss (teaches sharp relative ordering on 2-ply leaves!)
        try:
            xpb, ypb = next(pair_iter)
        except StopIteration:
            pair_iter = iter(pair_loader)
            xpb, ypb = next(pair_iter)
        xpb, ypb = xpb.to(device), ypb.to(device)
        lp = model(xpb.view(-1, 6, 8, 8), quantize=use_qat).view(-1, 2)
        pp = torch.sigmoid(lp)
        loss_pair_wdl = F.binary_cross_entropy_with_logits(lp, ypb)
        gap = ypb[:, 0] - ypb[:, 1]
        sign_t = torch.sign(gap)
        weight_pair = (gap.abs() >= 0.04).float()
        # Require logit difference to follow Stockfish's ordering with soft margin proportional to WDL gap
        margin = torch.clamp(gap.abs() * 3.2, max=1.5)
        loss_rank = (weight_pair * F.relu(margin - sign_t * (lp[:, 0] - lp[:, 1]))).mean()

        loss = loss_wdl + 0.45 * loss_pair_wdl + 0.35 * loss_rank
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.5)
        optimizer.step()

        run_loss += loss.item()
        steps += 1

    scheduler.step()
    val_mae, val_corr, val_pacc = evaluate_value_net(val_loader, xp_val_f, yp_val_f, quantize=True)
    history["train_loss"].append(run_loss / max(1, steps))
    history["val_mae"].append(val_mae)
    history["val_corr"].append(val_corr)
    history["val_pair_acc"].append(val_pacc)

    # Composite selection metric: high Stockfish correlation + high sibling ranking accuracy - MAE
    score = val_corr + val_pacc - 2.0 * val_mae
    marker = ""
    if use_qat and score >= best_score:
        best_score = score
        best_state = deepcopy(model.state_dict())
        best_metrics = (val_mae, val_corr, val_pacc)
        marker = " * [best]"

    mode_s = "8-bit QAT" if use_qat else "Warmup   "
    print(
        f"Epoch {epoch:02d}/{EPOCHS:02d} ({mode_s}) | "
        f"Loss: {history['train_loss'][-1]:.4f} | "
        f"Val WDL MAE: {val_mae:.4f} | "
        f"SF Corr r: {val_corr:.4f} | "
        f"Pair Rank Acc: {val_pacc * 100:5.2f}%{marker}"
    )

model.load_state_dict(best_state)
print(
    f"\nLoaded best QAT checkpoint -> Val MAE: {best_metrics[0]:.4f} | "
    f"Stockfish r: {best_metrics[1]:.4f} | Pairwise Ranking: {best_metrics[2] * 100:.2f}%"
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.4))
ep_x = range(1, EPOCHS + 1)
ax1.plot(ep_x, history["train_loss"], label="Train Distillation Loss")
ax1.plot(ep_x, history["val_mae"], label="Val WDL MAE")
ax1.set_xlabel("Epoch")
ax1.set_title("Loss & WDL Calibration Error")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep_x, [c * 100 for c in history["val_corr"]], label="Stockfish Pearson r (%)")
ax2.plot(ep_x, [p * 100 for p in history["val_pair_acc"]], label="Sibling Move Rank Acc (%)")
ax2.set_xlabel("Epoch")
ax2.set_title("8-Bit QAT Correlation & Ranking Accuracy")
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 5. Compile Trained Model to SVG `<feConvolveMatrix>` & `<feColorMatrix>` Spec
#    + Bit-Exact 8-Bit NumPy SVG Simulator & 2-Ply Minimax Engine
# ---------------------------------------------------------------------------

def flip_kernel_2d(k2d: np.ndarray) -> List[float]:
    """Flip 2D spatial kernel 180 degrees so W3C `<feConvolveMatrix>` evaluates exact cross-correlation."""
    return [round(float(v), 6) for v in k2d[::-1, ::-1].reshape(-1)]


def export_depthwise_conv(conv: nn.Conv2d) -> dict:
    w = conv.weight.detach().cpu().numpy()  # (C, 1, kH, kW)
    b = conv.bias.detach().cpu().numpy()    # (C,)
    kh, kw = int(w.shape[2]), int(w.shape[3])
    tx = (kw - 1) // 2 if conv.padding[1] > 0 else 0
    ty = (kh - 1) // 2 if conv.padding[0] > 0 else 0
    filters = []
    for c in range(w.shape[0]):
        filters.append({
            "channel": int(c),
            "order": [kw, kh],
            "targetX": int(tx),
            "targetY": int(ty),
            "divisor": 1.0,
            "bias": round(float(b[c]), 6),
            "kernelMatrix": flip_kernel_2d(w[c, 0]),
            "padding": int(conv.padding[0]),
        })
    return {
        "num_channels": int(w.shape[0]),
        "svg_order": [kw, kh],
        "targetX": int(tx),
        "targetY": int(ty),
        "padding": int(conv.padding[0]),
        "filters": filters,
    }


def export_pointwise(pw: SVGPointwise1x1) -> dict:
    w = pw.weight.detach().cpu().numpy()
    b = pw.bias.detach().cpu().numpy()
    s_ch = pw.compute_channel_scales().cpu().numpy()
    z0 = ZERO_POINT
    g_in = pw.in_channels // 3
    g_out = pw.out_channels // 3
    triplets = []
    for o in range(g_out):
        s_vec = [float(s_ch[3 * o + r]) for r in range(3)]
        b_slice = b[3 * o : 3 * (o + 1)]
        color_matrices = []
        for g in range(g_in):
            blk = w[3 * o : 3 * (o + 1), 3 * g : 3 * (g + 1)] / np.array(s_vec, dtype=np.float64)[:, None]
            mat20 = [
                round(float(blk[0, 0]), 6), round(float(blk[0, 1]), 6), round(float(blk[0, 2]), 6), 0.0, z0,
                round(float(blk[1, 0]), 6), round(float(blk[1, 1]), 6), round(float(blk[1, 2]), 6), 0.0, z0,
                round(float(blk[2, 0]), 6), round(float(blk[2, 1]), 6), round(float(blk[2, 2]), 6), 0.0, z0,
                0.0, 0.0, 0.0, 0.0, 1.0,
            ]
            color_matrices.append(mat20)
        triplets.append({
            "out_triplet": int(o),
            "scales": [round(s, 6) for s in s_vec],
            "rescale_slopes": [round(s, 6) for s in s_vec],
            "rescale_intercepts": [round(float(b_slice[r] - z0 * s_vec[r]), 6) for r in range(3)],
            "color_matrices": color_matrices,
        })
    return {"in_channels": pw.in_channels, "out_channels": pw.out_channels, "zero_point": z0, "triplets": triplets}


def export_linear_skip(skip: SVGLinearSkip6to3) -> dict:
    w = skip.effective_weight().detach().cpu().numpy()  # (3, 6)
    z0 = ZERO_POINT
    mats = []
    for g in range(2):
        blk = w[:, 3 * g : 3 * (g + 1)]
        offs = [round(float(z0 - blk[r].sum() * z0), 6) for r in range(3)]
        mats.append([
            round(float(blk[0, 0]), 6), round(float(blk[0, 1]), 6), round(float(blk[0, 2]), 6), 0.0, offs[0],
            round(float(blk[1, 0]), 6), round(float(blk[1, 1]), 6), round(float(blk[1, 2]), 6), 0.0, offs[1],
            round(float(blk[2, 0]), 6), round(float(blk[2, 1]), 6), round(float(blk[2, 2]), 6), 0.0, offs[2],
            0.0, 0.0, 0.0, 0.0, 1.0,
        ])
    return {
        "zero_point": z0,
        "weights_3x6": [[round(float(x), 6) for x in row] for row in w],
        "color_matrices": mats,
        "sum_stages": [
            {"order": [2, 2], "divisor": 1.25, "bias": round(-2.2 * z0, 6), "kernelMatrix": [1, 1, 1, 1]},
            {"order": [2, 2], "divisor": 1.60, "bias": round(-1.5 * z0, 6), "kernelMatrix": [1, 1, 1, 1]},
            {"order": [2, 2], "divisor": 1.00, "bias": round(-3.0 * z0, 6), "kernelMatrix": [1, 1, 1, 1]},
        ],
    }


def export_value_head(model: SVGChessValueCNN) -> dict:
    w_fc = model.fc.weight.detach().cpu().numpy()[0]  # (27,)
    b_fc = float(model.fc.bias.detach().cpu().item())
    z0 = ZERO_POINT
    m_half = float(model.compute_logit_half_range().cpu().item())
    two_m = 2.0 * m_half
    g_in = len(w_fc) // 3

    color_matrices = []
    for g in range(g_in):
        sl = w_fc[3 * g : 3 * (g + 1)] / two_m
        if g == 0:
            off = z0 + b_fc / two_m
        elif g == g_in - 1:
            # Skip triplet (g == 8) is centered around Z0!
            off = z0 - float(sl.sum()) * z0
        else:
            off = z0
        row = [round(float(sl[0]), 6), round(float(sl[1]), 6), round(float(sl[2]), 6), 0.0, round(float(off), 6)]
        color_matrices.append(row + row + row + [0.0, 0.0, 0.0, 0.0, 1.0])

    # 256-entry `<feComponentTransfer type="table">` Sigmoid LUT mapping `acc` into `[3/255 .. 253/255]`
    # (`0.50` WDL = exact `128/255 = Z0`; `0` = invalid move, `254` = Mate-in-1, `255` = empty reply mask)
    grid = np.arange(256, dtype=np.float64) / 255.0
    logits_grid = (grid - z0) * two_m
    sig_probs = 1.0 / (1.0 + np.exp(-np.clip(logits_grid, -20.0, 20.0)))
    v_table_u8 = np.clip(np.round(3.0 + 250.0 * sig_probs), 3.0, 253.0) / 255.0

    return {
        "in_features": int(len(w_fc)),
        "zero_point": z0,
        "u8_min": 3,
        "u8_max": 253,
        "logit_half_range_M": round(m_half, 6),
        "color_matrices": color_matrices,
        "sigmoid_table_values": [round(float(v), 6) for v in v_table_u8],
    }


def build_chess_export_spec(model: SVGChessValueCNN) -> dict:
    return {
        "format": "nojs-svg-chess-value-cnn-v1",
        "arch_version": 1,
        "zero_point": ZERO_POINT,
        "zero_point_u8": ZERO_POINT_U8,
        "pos_val_u8": POS_VAL_U8,
        "neg_val_u8": NEG_VAL_U8,
        "input_shape": [6, 8, 8],
        "channel_order": ["PAWN", "KNIGHT", "BISHOP", "ROOK", "QUEEN", "KING"],
        "layers": {
            "mix1": export_pointwise(model.mix1),
            "dwconv1": export_depthwise_conv(model.dwconv1),
            "pool1": {"order": [2, 2], "targetX": 0, "targetY": 0, "divisor": 4.0, "kernelMatrix": [1, 1, 1, 1]},
            "mix2": export_pointwise(model.mix2),
            "dwconv2": export_depthwise_conv(model.dwconv2),
            "pool2": {"order": [2, 2], "targetX": 0, "targetY": 0, "divisor": 4.0, "kernelMatrix": [1, 1, 1, 1]},
            "mix3": export_pointwise(model.mix3),
            "dwconv3": export_depthwise_conv(model.dwconv3),
            "skip_mat": export_linear_skip(model.skip_mat),
            "value_head": export_value_head(model),
        },
    }


# --- Pure-NumPy Bit-Exact 8-Bit SVG `<filter>` Value Simulator ---

def q8_np(x: np.ndarray) -> np.ndarray:
    return np.round(np.clip(x, 0.0, 1.0) * 255.0) / 255.0


def svg_lut_eval_np(x: np.ndarray, table_vals: Sequence[float]) -> np.ndarray:
    tbl = np.asarray(table_vals, dtype=np.float64)
    n = len(tbl) - 1
    pos = np.clip(x, 0.0, 1.0) * n
    idx = np.minimum(np.floor(pos).astype(np.int64), n - 1)
    frac = pos - idx
    return q8_np(tbl[idx] * (1.0 - frac) + tbl[idx + 1] * frac)


def svg_fe_convolve_batch_np(buf: np.ndarray, order: Sequence[int], tx: int, ty: int,
                             kernel_flat: Sequence[float], divisor: float, bias: float, stride: int = 1) -> np.ndarray:
    ow, oh = int(order[0]), int(order[1])
    k = np.asarray(kernel_flat, dtype=np.float64).reshape(oh, ow)
    bsz, h, w = buf.shape
    padded = np.pad(buf, ((0, 0), (ty, oh - 1 - ty), (tx, ow - 1 - tx)), mode="constant", constant_values=0.0)
    acc = np.zeros((bsz, h, w), dtype=np.float64)
    for i in range(oh):
        for j in range(ow):
            kv = k[oh - 1 - i, ow - 1 - j]
            if kv != 0.0:
                acc += kv * padded[:, i : i + h, j : j + w]
    res = q8_np(acc / float(divisor) + float(bias))
    if stride > 1:
        return res[:, 0::stride, 0::stride]
    if oh == 2 and ow == 2 and tx == 0 and ty == 0 and h == 2 and w == 2:
        return res[:, 0:1, 0:1]
    return res


def svg_pointwise_batch_np(ch_bufs: np.ndarray, pw_spec: dict) -> np.ndarray:
    bsz, _, h, w = ch_bufs.shape
    z0 = float(pw_spec["zero_point"])
    out = np.zeros((bsz, pw_spec["out_channels"], h, w), dtype=np.float64)
    for trip in pw_spec["triplets"]:
        o = trip["out_triplet"]
        acc = None
        for g, mat20 in enumerate(trip["color_matrices"]):
            m = np.asarray(mat20, dtype=np.float64).reshape(4, 5)[:3, :]
            rgb_in = ch_bufs[:, 3 * g : 3 * (g + 1)]
            part = q8_np(np.einsum("rc,bchw->brhw", m[:, :3], rgb_in) + m[:, 4].reshape(1, 3, 1, 1))
            acc = part if acc is None else q8_np(acc + part - z0)
        slopes = np.asarray(trip["rescale_slopes"], dtype=np.float64).reshape(1, 3, 1, 1)
        intercepts = np.asarray(trip["rescale_intercepts"], dtype=np.float64).reshape(1, 3, 1, 1)
        out[:, 3 * o : 3 * (o + 1)] = q8_np(acc * slopes + intercepts)
    return out


def simulate_svg_value_batch(x_float_6x8x8: np.ndarray, spec: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Bit-exact 8-bit SVG `<filter>` simulation on `(N, 6, 8, 8)` inputs.
    Returns `(decoded_logits, calibrated_probs, svg_byte_scores_2_to_253)`.
    """
    ly = spec["layers"]
    x0 = q8_np(x_float_6x8x8)

    # Linear material/phase skip branch (8x8 -> 4x4 -> 2x2 -> 1x1)
    sk_spec = ly["skip_mat"]
    z0 = float(sk_spec["zero_point"])
    m0 = np.asarray(sk_spec["color_matrices"][0], dtype=np.float64).reshape(4, 5)[:3, :]
    m1 = np.asarray(sk_spec["color_matrices"][1], dtype=np.float64).reshape(4, 5)[:3, :]
    p0 = q8_np(np.einsum("rc,bchw->brhw", m0[:, :3], x0[:, 0:3]) + m0[:, 4].reshape(1, 3, 1, 1))
    p1 = q8_np(np.einsum("rc,bchw->brhw", m1[:, :3], x0[:, 3:6]) + m1[:, 4].reshape(1, 3, 1, 1))
    sk = q8_np(p0 + p1 - z0)
    for stg in sk_spec["sum_stages"]:
        stride = 2 if sk.shape[-1] > 2 else 1
        sk = np.stack([
            svg_fe_convolve_batch_np(sk[:, c], stg["order"], 0, 0, stg["kernelMatrix"], stg["divisor"], stg["bias"], stride=stride)
            for c in range(3)
        ], axis=1)

    # Layer 1 (8x8 -> 4x4)
    h = svg_pointwise_batch_np(x0, ly["mix1"])
    h = np.stack([
        svg_fe_convolve_batch_np(h[:, f["channel"]], f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"])
        for f in ly["dwconv1"]["filters"]
    ], axis=1)
    p1_s = ly["pool1"]
    h = np.stack([
        svg_fe_convolve_batch_np(h[:, c], p1_s["order"], 0, 0, p1_s["kernelMatrix"], p1_s["divisor"], 0.0, stride=2)
        for c in range(h.shape[1])
    ], axis=1)

    # Layer 2 (4x4 -> 2x2)
    h = svg_pointwise_batch_np(h, ly["mix2"])
    h = np.stack([
        svg_fe_convolve_batch_np(h[:, f["channel"]], f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"])
        for f in ly["dwconv2"]["filters"]
    ], axis=1)
    p2_s = ly["pool2"]
    h = np.stack([
        svg_fe_convolve_batch_np(h[:, c], p2_s["order"], 0, 0, p2_s["kernelMatrix"], p2_s["divisor"], 0.0, stride=2)
        for c in range(h.shape[1])
    ], axis=1)

    # Layer 3 (2x2 -> 1x1)
    h = svg_pointwise_batch_np(h, ly["mix3"])
    h = np.stack([
        svg_fe_convolve_batch_np(h[:, f["channel"]], f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"])
        for f in ly["dwconv3"]["filters"]
    ], axis=1)

    feat = np.concatenate([h[:, :, 0, 0], sk[:, :, 0, 0]], axis=1)  # (N, 27)
    vh = ly["value_head"]
    z0_vh = float(vh["zero_point"])
    acc = None
    for g, mat20 in enumerate(vh["color_matrices"]):
        m = np.asarray(mat20, dtype=np.float64).reshape(4, 5)[0, :]
        part = q8_np(feat[:, 3 * g : 3 * (g + 1)] @ m[:3] + m[4])
        acc = part if acc is None else q8_np(acc + part - z0_vh)

    two_m = 2.0 * float(vh["logit_half_range_M"])
    decoded_logits = (acc - z0_vh) * two_m
    v_svg = svg_lut_eval_np(acc, vh["sigmoid_table_values"])
    v_u8 = np.round(v_svg * 255.0).astype(np.int32)
    probs = (v_svg * 255.0 - 3.0) / 250.0
    return decoded_logits, probs, v_u8


export_spec = build_chess_export_spec(model)
print("Compiled SVG `<filter>` Chess Value CNN export spec successfully!")
print(f"  Logit half-range M = +-{export_spec['layers']['value_head']['logit_half_range_M']}")
print(f"  Linear skip row 0  = {export_spec['layers']['skip_mat']['weights_3x6'][0]}")


In [ ]:
# ---------------------------------------------------------------------------
# 6. Bit-Exact 8-Bit SVG Parity Check & 2-Ply CNN Minimax Tactical Verification
# ---------------------------------------------------------------------------

@torch.no_grad()
def get_pytorch_qat_logits(x_np: np.ndarray) -> np.ndarray:
    model.eval()
    outs = []
    for i in range(0, len(x_np), 512):
        xb = torch.from_numpy(x_np[i : i + 512]).to(device)
        outs.append(model(xb, quantize=True).cpu().numpy())
    return np.concatenate(outs, axis=0)


qat_logits_test = get_pytorch_qat_logits(x_test_f)
qat_probs_test = 1.0 / (1.0 + np.exp(-np.clip(qat_logits_test, -20.0, 20.0)))
svg_logits_test, svg_probs_test, svg_u8_test = simulate_svg_value_batch(x_test_f, export_spec)

logit_abs_diff = np.abs(qat_logits_test - svg_logits_test)
lsb_step = (2.0 * float(export_spec["layers"]["value_head"]["logit_half_range_M"])) / 255.0
lsb_diff = np.round(logit_abs_diff / max(lsb_step, 1e-9))

test_mae_qat = float(np.mean(np.abs(qat_probs_test - y_test_f)))
test_corr_qat = float(np.corrcoef(qat_probs_test, y_test_f)[0, 1])
test_mae_svg = float(np.mean(np.abs(svg_probs_test - y_test_f)))
test_corr_svg = float(np.corrcoef(svg_probs_test, y_test_f)[0, 1])
qat_svg_mae_gap = abs(test_mae_qat - test_mae_svg)
qat_svg_corr_gap = abs(test_corr_qat - test_corr_svg)

# Evaluate 2-Ply SVG CNN Minimax (matching exact page destination-bucket + LVA Ply-2 semantics!)
LVA_ORDER = {chess.PAWN: 1, chess.KNIGHT: 2, chess.BISHOP: 3, chess.ROOK: 4, chess.QUEEN: 5, chess.KING: 6}

def select_move_2ply_svg_minimax(board: chess.Board, spec: dict) -> Tuple[chess.Move, int]:
    """Exact 2-Ply Minimax using the 8-bit SVG CNN Value Network (`erode` MIN over legal opponent
    destination buckets, `dilate` MAX over AI legal destination buckets, with `254` = Mate-in-1,
    `128` = Stalemate/Draw, and `255` = empty reply slot identity).
    """
    ai_col = board.turn
    # 1. Ply-1 LVA legal move per destination square v1 in 0..63 (matches CSS `.e12` `--m1_src_0..63`)
    m1_by_dst: Dict[int, chess.Move] = {}
    for mv in board.legal_moves:
        if mv.promotion not in (None, chess.QUEEN):
            continue
        cur = m1_by_dst.get(mv.to_square)
        if cur is None or LVA_ORDER[board.piece_type_at(mv.from_square)] < LVA_ORDER[board.piece_type_at(cur.from_square)]:
            m1_by_dst[mv.to_square] = mv

    q_v1: Dict[int, int] = {v1: 255 for v1 in m1_by_dst}
    leaf_boards_u8: List[np.ndarray] = []
    leaf_map: List[Tuple[int, int]] = []  # (v1, v2)

    for v1, m1 in m1_by_dst.items():
        b1 = board.copy(stack=False)
        b1.push(m1)
        if b1.is_checkmate():
            q_v1[v1] = 254
            continue
        if b1.is_stalemate() or b1.is_insufficient_material():
            q_v1[v1] = ZERO_POINT_U8
            continue

        # Opponent's LVA legal reply per target square v2 in 0..63
        m2_by_dst: Dict[int, chess.Move] = {}
        for m2 in b1.legal_moves:
            if m2.promotion not in (None, chess.QUEEN):
                continue
            cur2 = m2_by_dst.get(m2.to_square)
            if cur2 is None or LVA_ORDER[b1.piece_type_at(m2.from_square)] < LVA_ORDER[b1.piece_type_at(cur2.from_square)]:
                m2_by_dst[m2.to_square] = m2

        for v2, m2 in m2_by_dst.items():
            b2 = b1.copy(stack=False)
            b2.push(m2)
            leaf_boards_u8.append(encode_board_uint8(b2, ai_color=ai_col))
            leaf_map.append((v1, v2))

    if leaf_boards_u8:
        x_leaves = np.stack(leaf_boards_u8, axis=0).astype(np.float32) / 255.0
        _, _, u8_scores = simulate_svg_value_batch(x_leaves, spec)
        for idx, (v1, _v2) in enumerate(leaf_map):
            val = int(u8_scores[idx])
            if val < q_v1[v1]:
                q_v1[v1] = val

    best_v1 = max(q_v1.keys(), key=lambda v: (q_v1[v], -abs((v % 8) - 3.5) - abs((v // 8) - 3.5)))
    return m1_by_dst[best_v1], q_v1[best_v1]


TACTICAL_SUITE = [
    (
        "Capture Hanging Queen",
        "r1bqkbnr/ppp2ppp/2np4/4p3/4P1Q1/2N2N2/PPPP1PPP/R1B1KB1R b KQkq - 1 4",
        {"c8g4"},  # Bxg4 wins White's undefended Queen on g4!
    ),
    (
        "Refuse Defended-Pawn Bait (Do NOT blunder Queen!)",
        "rnb1kbnr/pppp1ppp/8/4p3/4P2q/2N2N2/PPPP1PPP/R1BQKB1R b KQkq - 3 3",
        None,     # Black Queen on h4 is attacked by Nf3 and must NOT play Qxe4?? (Nxe4 loses Q) or stay on h4!
    ),
    (
        "Deliver Mate in 1 (Back-Rank Mate)",
        "6k1/5ppp/8/8/8/8/5PPP/3R2K1 w - - 0 1",
        {"d1d8"},  # Rd8#
    ),
    (
        "Win Undefended Rook",
        "4k3/p7/8/3b4/8/8/8/4K2R b - - 0 1",
        {"d5h1"},  # Bxh1 wins White's undefended Rook on h1!
    ),
    (
        "Avoid Stalemate Trap & Deliver Mate",
        "7k/8/5K2/6Q1/8/8/8/8 w - - 0 1",
        {"g5g7"},  # Qg7# (254) beats Qg6?? stalemate (128)!
    ),
]

print("=====================================================================")
print(f"  Held-Out Test Set Size:                 {len(y_test_f):,} unseen Stockfish positions")
print(f"  PyTorch 8-Bit QAT WDL MAE / Corr r:     {test_mae_qat:.4f} / {test_corr_qat:.4f}")
print(f"  Exact 8-Bit SVG WDL MAE / Corr r:       {test_mae_svg:.4f} / {test_corr_svg:.4f}  (gap: {qat_svg_mae_gap:.4f} / {qat_svg_corr_gap:.4f})")
print(f"  FP32-QAT vs FP64-SVG Byte Diff (LSBs):  Median {np.median(lsb_diff):.0f} LSB | 99th pct {np.percentile(lsb_diff, 99):.0f} LSB | Max {lsb_diff.max():.0f} LSB (1 LSB = {lsb_step:.4f} logits)")
print("=====================================================================")
assert test_corr_svg >= 0.82, f"Expected >= 0.82 Stockfish Pearson r, got {test_corr_svg:.4f}"
assert qat_svg_mae_gap < 0.008 and qat_svg_corr_gap < 0.008, (
    f"Expected near-zero QAT-to-SVG gap, got MAE gap {qat_svg_mae_gap:.4f}, r gap {qat_svg_corr_gap:.4f}"
)
assert np.median(lsb_diff) <= 2, f"Expected median <= 2 LSB difference, got {np.median(lsb_diff)}"

print("\n2-Ply SVG CNN Minimax Tactical Sanity Checks:")
for name, fen, expected_ucis in TACTICAL_SUITE:
    bd = chess.Board(fen)
    mv, sc = select_move_2ply_svg_minimax(bd, export_spec)
    if expected_ucis is not None:
        status_s = "PASS" if mv.uci() in expected_ucis else f"WARN (got {mv.uci()})"
    else:
        # Queen on h4 attacked by Nf3: verify Black moves the Queen off h4 to a safe square!
        b_after = bd.copy(); b_after.push(mv)
        q_safe = (mv.from_square == chess.H4) and not b_after.is_attacked_by(chess.WHITE, mv.to_square)
        status_s = "PASS (Queen saved cleanly!)" if q_safe else f"WARN (played {mv.uci()})"
    print(f"  [{status_s}] {name:48s} -> {mv.uci()} (score_u8={sc}/255)")


In [ ]:
# ---------------------------------------------------------------------------
# 7. Export `weights.json` (with Golden SVG Test Vectors) & Trigger Download
# ---------------------------------------------------------------------------

golden_positions = []
for idx in range(24):
    fen_i = SINGLE_FENS[te_idx[idx]]
    golden_positions.append({
        "id": f"chess_test_{idx:02d}",
        "fen": fen_i,
        "regime": regimes_test[idx],
        "stockfish_wdl": round(float(y_test_f[idx]), 5),
        "svg_u8": int(svg_u8_test[idx]),
        "svg_prob": round(float(svg_probs_test[idx]), 5),
        "svg_logit": round(float(svg_logits_test[idx]), 4),
    })

export_spec["metrics"] = {
    "val_wdl_mae_qat": round(float(best_metrics[0]), 5),
    "val_stockfish_corr_r": round(float(best_metrics[1]), 5),
    "val_sibling_rank_acc": round(float(best_metrics[2]), 5),
    "test_wdl_mae_svg_uint8": round(float(test_mae_svg), 5),
    "test_stockfish_corr_svg_uint8": round(float(test_corr_svg), 5),
    "qat_to_svg_median_lsb_diff": int(round(float(np.median(lsb_diff)))),
    "qat_to_svg_max_lsb_diff": int(round(float(lsb_diff.max()))),
    "qat_to_svg_max_logit_diff": round(float(logit_abs_diff.max()), 6),
    "total_parameters": int(total_params),
    "num_labeled_positions": int(len(combined_fens)),
}
export_spec["golden_positions"] = golden_positions

OUT_PATH = "weights.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(export_spec, f, separators=(",", ":"))

size_kb = os.path.getsize(OUT_PATH) / 1024.0
print(
    f"Saved {OUT_PATH} ({size_kb:.1f} KB) | "
    f"Test Stockfish r = {test_corr_svg:.4f} | Test MAE = {test_mae_svg:.4f}"
)

try:
    from google.colab import files
    files.download(OUT_PATH)
    print("Triggered browser download for weights.json!")
except ImportError:
    print("Not running inside Google Colab; weights.json written to current directory.")
